# Import sempy and Fabric libraries

In [ ]:
import sempy
import sempy.fabric as fabric
import pandas

## Import Parameters

In [ ]:
%run 2. Parameters

## Attach Lakehouse

In [ ]:
import notebookutils

workspace_id = fabric.get_workspace_id()

# lakehouse_name is already available from %run above
lh = notebookutils.lakehouse.get(lakehouse_name, workspace_id)
lakehouse_id = lh.id

spark.conf.set("spark.microsoft.defaultLakehouseId", lakehouse_id)
spark.conf.set("spark.microsoft.defaultLakehouseWorkspaceId", workspace_id)

print(f"✅ Lakehouse '{lakehouse_name}' ({lakehouse_id}) attached in workspace '{workspace_id}'")

In [ ]:
# Check current lakehouse context
context = notebookutils.runtime.context
print(f"Lakehouse ID:           {context.get('defaultLakehouseId')}")
print(f"Lakehouse Name:         {context.get('defaultLakehouseName')}")
print(f"Lakehouse Workspace ID: {context.get('defaultLakehouseWorkspaceId')}")


## Setup Workspace and Dataset, then Run DAX Query


In [ ]:
df = fabric.evaluate_dax(datasetId, dax_string=dax_query, workspace=workspaceId)
display(df)

## Clean Column Names and Save DataFrame as Spark Table

In [ ]:
df.columns = [col.strip("[]") for col in df.columns]
spark_df = spark.createDataFrame(df)
table_name = f"calcDependency"
spark_df.write.saveAsTable(name=table_name, mode="overwrite")

In [ ]:
from datetime import datetime

df = fabric.evaluate_dax(datasetId, dax_string=dax_query, workspace=workspaceId)

# Add timestamp column
df['TIMESTAMP'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Clean column names
df.columns = [col.strip("[]") for col in df.columns]

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Define table name
table_name = "calcDependency"

# Determine version number based on existing data
if spark.catalog.tableExists(table_name):
    # Read existing table to find the max version for this semantic model
    existing_df = spark.read.table(table_name)
    
    # Filter for current semantic model and get max version
    max_version = existing_df.filter(
        existing_df.DATABASE_VERSION.startswith(semantic_model_name)
    ).selectExpr("max(cast(regexp_extract(DATABASE_VERSION, 'V(\\\\d+)', 1) as int)) as max_ver").collect()[0]['max_ver']
    
    # Increment version, or start at 1 if this model hasn't been added yet
    next_version = (max_version + 1) if max_version is not None else 1
    write_mode = "append"
else:
    next_version = 1
    write_mode = "overwrite"

# Add version column
from pyspark.sql.functions import lit
spark_df = spark_df.withColumn('DATABASE_VERSION', lit(f"{semantic_model_name} | V{next_version}"))

# Write using the determined mode
spark_df.write.option("mergeSchema", "true").saveAsTable(name=table_name, mode=write_mode)

display(df)

# Display the Data

In [ ]:
df = spark.sql(f"SELECT * FROM {lakehouse_name}.calcdependency")
display(df)